# Send the intro message to target contacts we have never written to

Walks your first-degree connections, keeps the ones classified into a target
industry that no message has ever gone to, and sends each of them the text in
`templates/intro.md` at a human cadence.

### Why it is driven from LinkedIn's relation list, not from Firestore

`memberDistance` on a stored contact is whatever it was when the profile was
scraped, and it is never refreshed. Measured on the first real run: **32 of 173
candidates — 18% — are stored as second-degree despite being connections
today**, so a Firestore-side distance filter would silently drop nearly a fifth
of the campaign.

`GET /users/relations` is the only list that is both current and complete. It
also carries the `ACoAA` provider id the send endpoints take, which `analysis`
does not hold at all.

So Firestore answers *"is this person a target, and have we written to them?"*
and LinkedIn answers *"who can we actually message?"*.

### Three guards against sending a second copy

Each covers a window the others cannot see, which is why all three are checked:

| Guard | Source | Catches |
|---|---|---|
| an open conversation | LinkedIn, live | anything — including a chat *they* opened and we never answered |
| `sent_total` | `analysis`, refreshed by Phase A0 | contacts messaged before this campaign existed |
| `intro_sent_at` | `analysis`, written here | a re-run before the next message sync |

`sent_total` is **absent**, not zero, for someone never written to — 4,906 of
the 4,912 — and absent entirely for anyone messaged below the `--since` floor
of your last sync. It narrows the list; it does not protect it. The live chat
map is what protects it.

`intro_sent_at` is a separate field on purpose: `refresh_contact_stats` deletes
`sent_total` from any contact whose messages it cannot find, so a marker
written there would not survive the next sync.

### Before you run it

- **Do a dry run first.** `DRY_RUN = True` reads everything, sends nothing,
  marks nobody, and drops the full candidate list into `intro-candidates.csv`
  for review.
- **Phase A0 runs `messages-sync.py` for you**, so `sent_total` reflects today
  rather than whenever that script was last run by hand. It writes to Firestore
  even under `DRY_RUN`: it mirrors your own mailbox rather than acting on
  LinkedIn, and a dry run built on a stale `sent_total` would name the wrong
  candidates. Set `SYNC_MESSAGES_FIRST = False` to skip it.
- **This is a multi-day job.** Sends are capped by
  `UNIPILE_MAX_MESSAGES_PER_DAY` — 50 by default, 100 in this `.env`; the cell
  below prints the value actually in force. Sends are paced 20–40s apart with a
  random 2–5 minute break roughly every 10 sends, so a 100-send day is a little
  over an hour of wall clock. The run stops cleanly when the budget is spent and
  picks up tomorrow — every phase recomputes from durable state, so re-running
  is always safe.
- **Newest connections go first.** "Thanks for connecting" ages badly, and at 50
  sends a day against thousands of candidates the order decides who ever gets
  it.

In [1]:
# Send the intro message to target LinkedIn contacts via Unipile
import logging
import os
import subprocess
import sys
import time
from datetime import UTC, datetime, timedelta
from pathlib import Path

import polars as pl
from dotenv import load_dotenv
from google.cloud import firestore

from functions import select_intro_candidates
from lib.unipile import UnipileClient
from lib.unipile.errors import (
    AccountRestricted,
    BudgetExhausted,
    RateLimited,
    UnipileError,
)

load_dotenv()

# The cadence pauses for minutes at a time on purpose; log it so a long silence
# in Phase E reads as a break rather than a hang.
logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("lib.unipile").setLevel(logging.INFO)

# Initialize Firestore client with specific project ID, only if file is found.
# This assignment also overrides the dev container default of
# /etc/credentials.json, which does not exist in the image.
if os.path.isfile("vk-linkedin-master-service-account.json"):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
        "vk-linkedin-master-service-account.json"
    )

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Dry run: find everyone and report what WOULD be sent, without messaging
# anyone and without marking anyone in Firestore. Use this for the first run.
# Phase A0 is the one exception -- see the note on it below.
DRY_RUN = False

# Refresh the message history before selecting anyone. `sent_total` is written
# by messages-sync.py and by nothing else, so without this the campaign filters
# on whenever that script last happened to run.
SYNC_MESSAGES_FIRST = True

# Cap this run below the daily budget. None means "use the whole budget".
MAX_SEND = 50

# The message. Read as UTF-8 because the copy contains typographic apostrophes.
TEMPLATE_PATH = Path("templates/intro.md")

# Where the dry run writes the reviewable candidate list.
CANDIDATES_CSV = Path("intro-candidates.csv")

# Who counts as a target. Recovr sells to RCM and medical billing companies,
# pathology practices, and independent or physician-practice-owned labs.
# Hospital-owned labs are not a target, which is why the classifier splits
# laboratories by owner: "Medical Lab" already excludes them, and a lab run by a
# pathology practice is classified "Pathology" rather than "Medical Lab".
TARGET_INDUSTRIES = {"RCM", "Pathology", "Medical Lab",  "Physician Practice"}

# Narrow by seniority, e.g. {"Owner", "Executive", "VP", "Director"}.
# None sends to every level.
TARGET_SENIORITY = None

# Values of the hand-set `handling` field that hold a contact back, whatever
# their industry says. "exclude" is a decision never to write to them; "manual"
# reserves them for a personal message a templated intro would pre-empt.
# Matched after trimming and lowercasing. 85 contacts carry one today.
HANDLING_HOLDS = {"exclude", "manual"}

# Skip anyone we already have a conversation with, whoever opened it. Turning
# this off sends the intro into existing threads — including ones holding an
# unanswered message from them, which is the worst place for it.
SKIP_EXISTING_CHATS = True

MESSAGE = TEMPLATE_PATH.read_text(encoding="utf-8").strip()

print(f"Configuration ready. Message from {TEMPLATE_PATH} ({len(MESSAGE)} chars):")
print("─" * 78)
print(MESSAGE)
print("─" * 78)
if DRY_RUN:
    print("\n⚠️  DRY_RUN=True — nothing will be sent and nothing written to Firestore.")

Configuration ready. Message from templates/intro.md (356 chars):
──────────────────────────────────────────────────────────────────────────────
Thanks for connecting. If you ever have any questions about using AI for automatic recovery of medical claims denials, I’d be happy to help, even if it's just asking advice or bouncing some ideas off me. Let's stay in touch!

P.S. We make the best AI Denial Recovery product in the market, and it is also plug-and-play, in case you are interested.

-Vitali
──────────────────────────────────────────────────────────────────────────────


In [3]:
db = firestore.Client(project="vk-linkedin", database="linkedin")

analysis_ref = db.collection("analysis")

# The Unipile client is created without a `with` block because notebooks do not
# reliably run __exit__. The last cell closes it.
li = UnipileClient.from_env()

print(f"LinkedIn account: {li.account_id}")
print(f"Message cap: {li.settings.max_messages_per_day} per 24h (Phase A counts what is spent)")

LinkedIn account: ZIGT4FVWS4CCJze_MuVHCg
Message cap: 200 per 24h (Phase A counts what is spent)


In [4]:
# ── Phase A0: Refresh the message history ─────────────────────────────────────
#
# `sent_total` on `analysis` is written by messages-sync.py and by nothing else,
# so it is exactly as stale as the last time that script was run by hand.
# Running it here means Phase D filters on today's history.
#
# This writes to Firestore even under DRY_RUN, and that is deliberate: it
# mirrors your own mailbox rather than acting on LinkedIn, and a dry run working
# from a stale `sent_total` would report the wrong candidates -- the one thing a
# dry run exists to get right. DRY_RUN governs what this campaign does to the
# outside world: no messages sent, no `intro_sent_at` written.
#
# A failure here stops the notebook rather than quietly continuing on old data.

if SYNC_MESSAGES_FIRST:
    print("Phase A0: syncing messages...")
    print()
    sync = subprocess.run(
        [sys.executable, "messages_sync.py"],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    print(sync.stdout)
    if sync.stderr:
        print(sync.stderr)
    if sync.returncode != 0:
        raise RuntimeError(
            "messages-sync.py failed, so `sent_total` is stale and Phase D would "
            "select from an out-of-date history. Fix the error above and re-run — "
            "or set SYNC_MESSAGES_FIRST = False to continue without it, in which "
            "case Phase C's live chat map is what stops a double message."
        )
else:
    print("Phase A0: skipped (SYNC_MESSAGES_FIRST = False)")
    print("  `sent_total` is as fresh as the last messages-sync.py run, and absent")
    print("  entirely below that run's --since floor. Phase C still guards the send.")


Phase A0: syncing messages...

messages_sync
  floor:         2023-01-01
  stored before: 2020-01-03 15:34 .. 2026-09-10 17:47
  listing all chats to resolve 50 conversations...
  loading attendees...
  indexing contacts from 'extracted'...
    110,958 keys in 9s
  forward pass:  56 new
  backward pass: 0 older
  contacts:      56 joined, 0 unmatched, 2 reused
  stored after:  2020-01-03 15:34 .. 2026-09-11 13:26
  analysis:      2,034 contacts, 464 with replies, 52 changed
  [!] 330 message(s) with no contact -- not counted
  elapsed: 23.2s



In [5]:
# ── Phase A: Count the last 24 hours of sends from LinkedIn ───────────────────
#
# The client starts every run with empty counters — nothing is kept on disk.
# This is where the budget learns what has already been spent, and it asks
# LinkedIn rather than a local tally, so messages sent from the app or from
# another run of this notebook are counted too.
#
# Counting the 24 hours ending *now*, rather than everything since UTC midnight,
# paces the account evenly: allowance comes back gradually as the sends that
# spent it age out, instead of all at once at midnight.
#
# Count here and nowhere else. Recounting mid-run would erase this run's own
# sends from the tally while it is still spending against them.

WINDOW = timedelta(hours=24)
cutoff = datetime.now(UTC) - WINDOW

sent_recently = li.messaging.count_messages_sent_since(cutoff)
li.budget.reconcile(message=sent_recently)

print(f"Phase A: counted since {cutoff:%Y-%m-%d %H:%M} UTC (last 24h)")
print(f"  messages sent  {sent_recently:>4}  →  {li.budget.remaining('message'):>4} left")

Phase A: counted since 2026-09-10 15:56 UTC (last 24h)
  messages sent    67  →   133 left


In [6]:
# ── Phase B: Load every classified contact ────────────────────────────────────
#
# One stream of `analysis`, projected to the five fields the filter reads. The
# collection is the join table: its document ids are LinkedIn slugs, which is
# exactly what a relation's `public_identifier` gives us.
#
# ~2.1k documents are keyed by numeric member id instead, left over from the
# LinkedIn Helper era, and carry neither a public-id nor an li-hash-id. They
# cannot be joined to a relation at all, and are counted as unclassified in
# Phase D — silence is the safe direction for a contact whose industry we
# cannot confirm.

print("Phase B: streaming classified contacts from 'analysis'...")

contacts = {
    doc.id: doc.to_dict() or {}
    for doc in analysis_ref.select(
        ["industry", "seniority", "handling", "sent_total", "intro_sent_at"]
    ).stream()
}
# Count classified contacts by industry across the whole dataset, and tag which
# industries are in the active target set.
industry_counts = {}
for body in contacts.values():
    industry = (body.get("industry") or "").strip()
    if industry:
        industry_counts[industry] = industry_counts.get(industry, 0) + 1

target_industry_counts = {
    industry: count for industry, count in industry_counts.items()
    if industry in TARGET_INDUSTRIES
}

print("  industry breakdown (all classified contacts):")
for industry, count in sorted(industry_counts.items(), key=lambda kv: (-kv[1], kv[0])):
    status = "target" if industry in TARGET_INDUSTRIES else "non-target"
    print(f"    {industry:<18} {count:>5}  ({status})")

print(f"  target-industry contacts: {sum(target_industry_counts.values())}")
on_target = sum(
    1 for body in contacts.values() if (body.get("industry") or "") in TARGET_INDUSTRIES
)

print(f"  classified contacts:  {len(contacts)}")
print(f"  in a target industry: {on_target}")

Phase B: streaming classified contacts from 'analysis'...
  industry breakdown (all classified contacts):
    Pathology           3532  (target)
    Hospital            2206  (non-target)
    Medical Lab         1965  (target)
    Non-Healthcare      1732  (non-target)
    Other Healthcare    1628  (non-target)
    Pharma              1281  (non-target)
    Health IT            883  (non-target)
    RCM                  658  (target)
    Health Insurance Payer    60  (non-target)
    Physician Practice    54  (target)
  target-industry contacts: 6209
  classified contacts:  28635
  in a target industry: 6209


In [7]:
# ── Phase C: Map every open conversation ──────────────────────────────────────
#
# `attendee_provider_id` is on the chat itself, so one walk of the chat list
# builds the whole map. This is the guard that cannot go stale, and building it
# once is also what keeps the send path cheap: `send_to` would re-walk every
# conversation on the account before each message to rediscover the same id.

print("Phase C: listing conversations...")

chat_ids = {}
for chat in li.messaging.iter_chats():
    if chat.attendee_provider_id:
        chat_ids.setdefault(chat.attendee_provider_id, chat.id)

print(f"  contacts with an open conversation: {len(chat_ids)}")

Phase C: listing conversations...
  contacts with an open conversation: 3088


In [8]:
# ── Phase D: Walk the connections and pick the candidates ─────────────────────
#
# `iter_relations` is the authoritative first-degree list. Everything the filter
# needs now exists, so selection is a pure function — see
# `select_intro_candidates` in functions.py, and its tests.

print("Phase D: listing first-degree connections...")

relations = list(li.users.iter_relations())
print(f"  first-degree connections: {len(relations)}")

candidates, skipped = select_intro_candidates(
    relations,
    contacts,
    chat_ids,
    industries=TARGET_INDUSTRIES,
    seniorities=TARGET_SENIORITY,
    holds=HANDLING_HOLDS,
    skip_existing_chats=SKIP_EXISTING_CHATS,
)

REASONS = {
    "unclassified": "unclassified — run analysis.ipynb / new-contacts.ipynb",
    "off_target": "outside the target industry or seniority",
    "handling": "held back by hand — handling is exclude/manual",
    "already_messaged": "sent_total says we have written to them",
    "intro_already_sent": "this campaign already messaged them",
    "existing_chat": "a conversation is already open",
}

print(f"\n  candidates: {len(candidates)}")
for reason, count in skipped.items():
    print(f"  skipped {count:>6}  {REASONS[reason]}")

budget_left = li.budget.remaining("message")
planned = min(len(candidates), budget_left)
if MAX_SEND is not None:
    planned = min(planned, MAX_SEND)

print(f"\n  message budget left (24h): {budget_left}")
print(f"  will send this run:        {planned}")
if len(candidates) > planned:
    print(f"  remaining for later runs:  {len(candidates) - planned}")

Phase D: listing first-degree connections...
  first-degree connections: 3387

  candidates: 78
  skipped    668  unclassified — run analysis.ipynb / new-contacts.ipynb
  skipped   1350  outside the target industry or seniority
  skipped     74  held back by hand — handling is exclude/manual
  skipped    979  sent_total says we have written to them
  skipped    144  this campaign already messaged them
  skipped     94  a conversation is already open

  message budget left (24h): 133
  will send this run:        50
  remaining for later runs:  28


In [9]:
# ── Phase D2: Review the queue ────────────────────────────────────────────────
#
# The whole candidate list goes to CSV, not just this run's slice: the rest is
# who the next runs will reach, and that is worth seeing before the first send.

queue = candidates[:planned]

if candidates:
    frame = pl.DataFrame(candidates)
    frame.write_csv(CANDIDATES_CSV)
    print(f"Wrote {len(candidates)} candidates to {CANDIDATES_CSV}\n")

    print(frame.select(["doc_id", "industry", "seniority", "connected_at", "name"]).head(15))

    print("\nBy industry:")
    print(frame.group_by("industry").len().sort("len", descending=True))
else:
    print("No candidates. Nothing to review.")

Wrote 78 candidates to intro-candidates.csv

shape: (15, 5)
┌─────────────────────┬────────────────────┬───────────┬─────────────────────┬─────────────────────┐
│ doc_id              ┆ industry           ┆ seniority ┆ connected_at        ┆ name                │
│ ---                 ┆ ---                ┆ ---       ┆ ---                 ┆ ---                 │
│ str                 ┆ str                ┆ str       ┆ datetime[μs, UTC]   ┆ str                 │
╞═════════════════════╪════════════════════╪═══════════╪═════════════════════╪═════════════════════╡
│ marianne-schmidt-cp ┆ Physician Practice ┆ Manager   ┆ 2026-09-11 01:37:44 ┆ Marianne Schmidt,   │
│ c-cpma-1942…        ┆                    ┆           ┆ UTC                 ┆ CPC, CPMA           │
│ kristine-venverloh- ┆ RCM                ┆ Manager   ┆ 2026-09-10 23:11:06 ┆ Kristine Venverloh  │
│ cpc-0687486…        ┆                    ┆           ┆ UTC                 ┆ , CPC               │
│ krishna-patel-70b1a ┆ Physici

In [10]:
# ── Phase E: Send ─────────────────────────────────────────────────────────────
#
# Every send waits first. The gap is drawn from a right-skewed distribution
# rather than a flat one, and a multi-minute break lands at random roughly every
# `long_pause_every` sends — LinkedIn watches the rhythm of actions, not only
# their count. Pacing and the daily cap are enforced inside `send_message` and
# `start_chat` themselves, so there is no way to send from here without going
# through them.
#
# `intro_sent_at` is written immediately after each success — both to Firestore
# and back onto the in-memory `contacts`, which is what makes THIS CELL safe to
# re-run on its own. Re-running it is the ordinary way to use a notebook, and
# `queue` is a snapshot from Phase D: without the local marker, a second
# execution would walk the same list and send every one of them a second copy.
# The daily cap does not save you there, because the sensible first live run is
# `MAX_SEND = 3`.
#
# The Firestore write is the same marker made durable, for the next run. If it
# ever fails the send still happened, so the failure is printed loudly — though
# the conversation it just opened would make Phase C skip them next run anyway.
# That is the point of having three guards.

s = li.settings

print(f"Phase E: sending to {len(queue)} contacts...")
print(f"  gap between sends: {s.min_delay_seconds:.0f}-{s.max_delay_seconds:.0f}s, skewed low")
if s.long_pause_every:
    print(
        f"  long break: every ~{s.long_pause_every} sends, "
        f"{s.long_pause_min_seconds:.0f}-{s.long_pause_max_seconds:.0f}s"
    )
else:
    print("  long break: disabled")

sent = []
send_failed = []
stopped = None

if DRY_RUN:
    print("\n  DRY_RUN=True — sending nothing. The first 3 would receive:\n")
    for row in queue[:3]:
        print(f"  → {row['name'] or row['doc_id']}  ({row['industry']} / {row['seniority']})")
        print(f"    {row['profile_url']}")
        print(f"    {'reply into ' + row['chat_id'] if row['chat_id'] else 'new conversation'}")
        print()
elif queue:
    progress_step = max(1, len(queue) // 10)

    for idx, row in enumerate(queue, 1):
        # Set below on every success, so a re-run of this cell skips them.
        if contacts[row["doc_id"]].get("intro_sent_at"):
            continue

        try:
            if row["chat_id"]:
                li.messaging.send_message(row["chat_id"], MESSAGE)
            else:
                li.messaging.start_chat([row["provider_id"]], MESSAGE)

        except BudgetExhausted as exc:
            stopped = f"daily budget spent after {idx - 1} sends: {exc.title}"
            print(f"  {stopped}")
            break
        except (RateLimited, AccountRestricted) as exc:
            # LinkedIn is limiting the account, not this message. Widen every
            # gap for whatever runs next in this process, then stop.
            li.budget.back_off()
            stopped = f"stopped after {idx - 1} sends: {exc.title}"
            print(f"\n  {stopped}")
            print(f"  {exc.detail}")
            break
        except UnipileError as exc:
            send_failed.append((row["doc_id"], exc.type or exc.title))
            continue

        sent.append(row)

        # Marked locally first: a Firestore write that fails must still stop
        # this session from sending to them again.
        contacts[row["doc_id"]]["intro_sent_at"] = True

        try:
            analysis_ref.document(row["doc_id"]).set(
                {"intro_sent_at": firestore.SERVER_TIMESTAMP}, merge=True
            )
        except Exception as exc:
            print(f"  ⚠️  SENT but not marked [{row['doc_id']}]: {exc}")

        if idx % progress_step == 0 or idx == len(queue):
            ts = time.strftime("%H:%M:%S")
            print(f"{ts} | sent {idx}/{len(queue)} ({idx * 100 // len(queue)}%)")

    print(f"\n  sent: {len(sent)},  failed: {len(send_failed)}")
    for doc_id, reason in send_failed[:5]:
        print(f"    error: {doc_id} → {reason}")
else:
    print("\n  Nothing to send.")

Phase E: sending to 50 contacts...
  gap between sends: 20-40s, skewed low
  long break: every ~10 sends, 120-300s
15:59:15 | sent 5/50 (10%)


Pausing 4m 11s to keep a human cadence.


16:05:26 | sent 10/50 (20%)
16:07:44 | sent 15/50 (30%)


Pausing 2m 05s to keep a human cadence.


16:09:55 | sent 20/50 (40%)


...

16:23:38 | sent 30/50 (60%)
16:25:52 | sent 35/50 (70%)


Pausing 3m 07s to keep a human cadence.
Pausing 2m 48s to keep a human cadence.


16:33:18 | sent 40/50 (80%)
16:35:46 | sent 45/50 (90%)


Pausing 3m 37s to keep a human cadence.


16:41:12 | sent 50/50 (100%)

  sent: 50,  failed: 0


In [11]:
# ── Phase F: Summary ──────────────────────────────────────────────────────────

remaining = len(candidates) - len(sent)

print("Run summary")
print(f"  first-degree connections:      {len(relations)}")
print(f"  candidates found:              {len(candidates)}")
print(f"  sent this run:                 {len(sent)}{' (dry run)' if DRY_RUN else ''}")
print(f"  failed:                        {len(send_failed)}")
print(f"  still to reach on later runs:  {remaining}")
print()
print(f"  message budget left (rolling 24h): {li.budget.remaining('message')}")

if stopped:
    print(
        "\n⚠️  The run stopped early. Wait several hours before the next one —"
        "\n    resuming immediately will just trip the same limit."
    )
elif remaining and not DRY_RUN:
    print(
        "\nRe-run this notebook once some of today's sends have aged past 24h to"
        "\n    continue. Phase A recounts the budget and Phase C picks up the"
        "\n    conversations this run opened, so nobody is messaged twice."
    )
if DRY_RUN:
    print(f"\n⚠️  This was a dry run. Review {CANDIDATES_CSV}, then set DRY_RUN = False.")

li.close()

Run summary
  first-degree connections:      3387
  candidates found:              78
  sent this run:                 50
  failed:                        0
  still to reach on later runs:  28

  message budget left (rolling 24h): 83

Re-run this notebook once some of today's sends have aged past 24h to
    continue. Phase A recounts the budget and Phase C picks up the
    conversations this run opened, so nobody is messaged twice.
